# 24-Hour Grid Optimization Analysis
This notebook generates 24 hours of environmental parameters, feeds them individually through the backend ML models, and passes the true predictive signals into the battery optimizer.

In [1]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
# Ensure root project folder is pathable to pick up modules
sys.path.append(os.getcwd())

from prediction.load import predict_load
from prediction.solar import predict_solar
from prediction.wind import predict_wind
from grid.cost_optimizer import run_grid_optimization

import numpy as np
import pandas as pd
import plotly.express as px


### 1. Define 24-Hour Environmental Feature Data
Here you can freely customize the hour-by-hour arrays to model specific weather patterns, solar conditions, and capacity constraints.

In [3]:
# Base static configuration
installed_solar_MW = 3000.0
panel_area_m2 = 12750000.0
num_panels = 7500000
installed_wind_MW = 50.0

# 24 arrays mapping specific hour conditions (Hour 0 to 23)
hours = np.arange(24)
times = [f'2025-10-13 {str(h).zfill(2)}:00:00' for h in hours]
weathers = ['Sunny'] * 24
temps_C = [22, 21, 20, 20, 21, 23, 25, 27, 30, 32, 34, 35, 36, 36, 35, 34, 31, 29, 27, 25, 24, 23, 23, 22]
humidities = [80, 82, 85, 85, 83, 75, 65, 55, 45, 40, 38, 35, 35, 36, 38, 42, 50, 60, 68, 72, 75, 78, 79, 80]
wind_speeds = [5.0, 4.7, 4.5, 4.6, 4.9, 5.5, 6.0, 6.5, 7.0, 7.5, 7.7, 7.5, 7.3, 7.0, 6.7, 6.3, 6.0, 5.3, 5.0, 4.7, 4.5, 4.6, 4.8, 4.9]
precips = [0.0] * 24

# Irradiance naturally rises at dawn and dies at dusk
irradiances = [0, 0, 0, 0, 0, 0, 50, 200, 450, 650, 800, 950, 1000, 950, 750, 500, 250, 50, 0, 0, 0, 0, 0, 0]


### 2. Connect 24-Hour Loop into ML Predictions

In [4]:
solar_preds = []
wind_preds = []
load_preds = []

for h in range(24):
    load_inputs = {
        'datetime': times[h], 'weather': weathers[h], 'temp_C': temps_C[h], 
        'humidity_%': humidities[h], 'wind_speed_m_s': wind_speeds[h], 
        'solar_irradiance_W_m2': irradiances[h], 'precip_mm': precips[h],
        'installed_solar_MW': installed_solar_MW, 'panel_area_m2': panel_area_m2, 
        'num_panels': num_panels, 'installed_wind_MW': installed_wind_MW
    }
    
    solar_inputs = {
        'datetime': times[h], 'weather': weathers[h], 'temp_C': temps_C[h], 
        'humidity_%': humidities[h], 'wind_speed_m_s': wind_speeds[h], 
        'solar_irradiance_W_m2': irradiances[h], 'precip_mm': precips[h],
        'installed_solar_MW': installed_solar_MW, 'panel_area_m2': panel_area_m2, 'num_panels': num_panels
    }
    
    wind_inputs = {
        'datetime': times[h], 'weather': weathers[h], 'temp_C': temps_C[h], 
        'humidity_%': humidities[h], 'wind_speed_m_s': wind_speeds[h], 
        'precip_mm': precips[h], 'installed_wind_MW': installed_wind_MW
    }
    
    # Execute ML functions
    load_preds.append(predict_load(load_inputs))
    solar_preds.append(predict_solar(solar_inputs))
    wind_preds.append(predict_wind(wind_inputs))

load_profile = np.array(load_preds)
solar_profile = np.array(solar_preds)
wind_profile = np.array(wind_preds)

print('24-Hour ML Predictions Extrapolated Successfully!')


24-Hour ML Predictions Extrapolated Successfully!


### 3. Run Mathematical Grid Optimizer and Charting

In [5]:
# Set your battery thresholds
battery_capacity = 1000  # MWh
max_power = 300  # MW

res = run_grid_optimization(load_profile, solar_profile, wind_profile, battery_capacity, max_power)

if res is None:
    print('Error: Optimizer infeasible constraints.')
else:
    df = res['df']
    
    def format_inr(value):
        abs_val = abs(value)
        if abs_val >= 1e7:
            return f'{value/1e7:,.2f} Cr'
        elif abs_val >= 1e5:
            return f'{value/1e5:,.2f} L'
        else:
            return f'{value:,.2f}'
            
    cost_str = f'Revenue: â‚¹ {format_inr(-res["total_cost"])}' if res['total_cost'] < 0 else f'Cost: â‚¹ {format_inr(res["total_cost"])}'
    print(f'\n--- Economic Performance ---')
    print(f'Total {cost_str}')
    print(f'Savings vs Grid-Only: â‚¹ {format_inr(res["savings"])}')
    print(f'Total Grid Import: {res["total_import"]:,.2f} MWh')



--- Economic Performance ---
Total Cost: â‚¹ 7.31 Cr
Savings vs Grid-Only: â‚¹ 72.41 L
Total Grid Import: 17,776.70 MWh


In [6]:
# Visualize Dispatch Timeline
chart_df = df[['Hour', 'Load (MW)', 'Solar (MW)', 'Wind (MW)', 'Grid Import (MW)']].set_index('Hour')
fig1 = px.line(chart_df, title='True 24-Hour Power Grid Dispatch Timeline')
fig1.show()

# Battery SoC
fig2 = px.line(df, x='Hour', y='SOC (%)', title='Battery Autonomy (%)')
fig2.show()


In [7]:
print('\n--- Hourly Dispatch Data ---')
display(df.style.highlight_max(axis=0))



--- Hourly Dispatch Data ---


,Hour,Load (MW),Solar (MW),Wind (MW),Net Load (MW),Battery Charge (MW),Battery Discharge (MW),SOC (%),Grid Import (MW),Grid Export (MW),Grid Tariff (â‚¹/MWh),Hourly Cost (â‚¹)
0,0,1186.490000,0.000000,1.510000,1184.980000,20.860000,0.000000,50.000000,1205.840000,0.000000,3500.000000,4220445.820000
1,1,1190.320000,0.000000,0.070000,1190.240000,27.040000,0.000000,51.980000,1217.290000,0.000000,3500.000000,4260503.900000
2,2,1191.330000,0.000000,0.000000,1191.330000,30.470000,0.000000,54.550000,1221.800000,0.000000,3500.000000,4276313.910000
3,3,1191.240000,0.000000,0.000000,1191.240000,33.740000,0.000000,57.450000,1224.980000,0.000000,3500.000000,4287434.180000
4,4,1186.080000,0.000000,0.740000,1185.340000,42.240000,0.000000,60.650000,1227.580000,0.000000,3500.000000,4296520.680000
5,5,1172.060000,0.000000,11.860000,1160.210000,89.530000,0.000000,64.660000,1249.730000,0.000000,3500.000000,4374065.240000
6,6,1187.010000,141.990000,39.910000,1005.110000,-0.000000,300.000000,73.170000,705.110000,-0.000000,7500.000000,5288294.380000
7,7,1199.320000,570.990000,96.440000,531.890000,-0.000000,300.000000,41.590000,231.890000,-0.000000,7500.000000,1739155.990000
8,8,1244.050000,1280.060000,185.360000,-221.370000,163.140000,-0.000000,10.010000,0.000000,58.230000,7500.000000,-145570.140000
9,9,1267.940000,1843.420000,322.010000,-897.480000,171.550000,-0.000000,25.510000,0.000000,725.930000,5500.000000,-1814832.770000
